# COMP5318 Assignment 1: Rice Classification

##### Group number: 149
##### Student 1 SID: 520510542
##### Student 2 SID: 510458096  
##### Student 3 SID: 520131808
##### Student 4 SID: 550749464

In [1]:
# Import all libraries
import sklearn
from sklearn.model_selection import StratifiedKFold

In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.

    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])



In [4]:
# Load the rice dataset: rice-final2.csv
import pandas as pd
import numpy as np
df = pd.read_csv("rice-final2.csv")
#df = pd.read_csv("test-before.csv")
#print(df.head())


In [5]:
# Pre-process dataset

#3. Changing the class values - The classes class1 and class2 should be changed to 0 and 1
#respectively.
df['class'] = df['class'].replace(['class1', 'class2'], [0, 1])

#1. Filling in the missing attribute values - The missing attribute values should be replaced with
#the mean value of the column using sklearn.impute.SimpleImputer.
from sklearn.impute import SimpleImputer

df = df.replace('?', np.nan)
mean_imputer = SimpleImputer(strategy="mean")
df = pd.DataFrame(mean_imputer.fit_transform(df), columns=df.columns)

#2. Normalising the data - Normalisation of each attribute should be performed using a min-max
#scaler to normalise the values between [0,1] with sklearn.preprocessing.MinMaxScaler.
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

#4. Print the first 10 rows of the pre-processed dataset. The feature values should be formatted to 4
#decimal places using .4f, the class value is an integer. A function print_data has been provided in
#the template to help you achieve this.

#print(df[:10])
for i in range(10):
  row_vals = []
  for j in range(len(df.columns)-1):
    val = df.iloc[i,j]
    row_vals.append(f"{val:.4f}")
  row_vals.append(int(df.iloc[i,len(df.columns)-1]))
  print(",".join(map(str, row_vals)))


0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


### Part 1: Cross-validation without parameter tuning

In [6]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers
from sklearn.model_selection import cross_val_score

In [7]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

def logregClassifier(X, y):
    lr = LogisticRegression(max_iter=10000, random_state=0)
    scores = cross_val_score(lr, X, y, cv=cvKFold)
    return scores.mean()

In [8]:
#Naïve Bayes
from sklearn.naive_bayes import GaussianNB

def nbClassifier(X, y):
  clf = GaussianNB()
  scores = cross_val_score(clf, X, y, cv=cvKFold)
  return scores.mean()

In [9]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

def dtClassifier(X, y):
  clf = DecisionTreeClassifier(random_state=0)
  scores = cross_val_score(clf, X, y, cv=cvKFold)
  return scores.mean()

In [10]:
# Ensembles: Bagging, Ada Boost and Gradient Boosting
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier

def bagDTClassifier(X, y, n_estimators, max_samples, max_depth):
    bagging_Classifier = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=max_depth, random_state=0), n_estimators=n_estimators, max_samples=max_samples, random_state=0)
    scores = cross_val_score(bagging_Classifier, X, y, cv=cvKFold)
    return scores.mean()

def adaDTClassifier(X, y, n_estimators, learning_rate, max_depth):
    ada_boost = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=max_depth, random_state=0), n_estimators=n_estimators, learning_rate=learning_rate, random_state=0)
    scores = cross_val_score(ada_boost, X, y, cv=cvKFold)
    return scores.mean()

def gbClassifier(X, y, n_estimators, learning_rate):
    gb_classifier = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, random_state=0)
    scores = cross_val_score(gb_classifier, X, y, cv=cvKFold)
    return scores.mean()

### Part 1 Results

In [11]:
# Parameters for Part 1:

#Bagging
bag_n_estimators = 50
bag_max_samples = 100
bag_max_depth = 5

#AdaBoost
ada_n_estimators = 50
ada_learning_rate = 0.5
ada_bag_max_depth = 5

#GB
gb_n_estimators = 50
gb_learning_rate = 0.5

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

LogR = logregClassifier(X, y)
NB = nbClassifier(X, y)
DT = dtClassifier(X, y)

Bagging = bagDTClassifier(X, y, bag_n_estimators, bag_max_samples, bag_max_depth)
AdaBoost = adaDTClassifier(X, y, ada_n_estimators, ada_learning_rate, ada_bag_max_depth)
GB = gbClassifier(X, y, gb_n_estimators, gb_learning_rate)

# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {LogR:.4f}")
print(f"NB average cross-validation accuracy: {NB:.4f}")
print(f"DT average cross-validation accuracy: {DT:.4f}")
print(f"Bagging average cross-validation accuracy: {Bagging:.4f}")
print(f"AdaBoost average cross-validation accuracy: {AdaBoost:.4f}")
print(f"GB average cross-validation accuracy: {GB:.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264
DT average cross-validation accuracy: 0.9057
Bagging average cross-validation accuracy: 0.9421
AdaBoost average cross-validation accuracy: 0.9314
GB average cross-validation accuracy: 0.9321


### Part 2: Cross-validation with parameter tuning

In [12]:
# KNN
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

k = [1, 3, 5, 7]
p = [1, 2]


def bestKNNClassifier(X_train, X_test, y_train, y_test):

    grid_search = GridSearchCV(KNeighborsClassifier(), param_grid={"n_neighbors": k, "p": p}, cv=cvKFold, return_train_score=True)
    grid_search.fit(X_train, y_train)

    best_params = grid_search.best_params_
    best_k = best_params["n_neighbors"]
    best_p = best_params["p"]
    best_cross_validation = grid_search.best_score_
    best_test_set = grid_search.score(X_test, y_test)

    return best_k, best_p, best_cross_validation, best_test_set

In [13]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]

def bestRFClassifier(X_train, X_test, y_train, y_test):
    grid_search = GridSearchCV(RandomForestClassifier(max_features="sqrt", random_state=0), param_grid={"n_estimators": n_estimators, "max_leaf_nodes": max_leaf_nodes}, cv=cvKFold, return_train_score=True)
    grid_search.fit(X_train, y_train)

    best_paras = grid_search.best_params_
    best_n_estimators = best_paras["n_estimators"]
    best_max_leaf_nodes = best_paras["max_leaf_nodes"]
    best_cross_validation = grid_search.best_score_
    best_test_set = grid_search.score(X_test, y_test)
    #print(best_n_estimators, best_max_leaf_nodes, best_cross_validation, best_test_set)
    y_pred = grid_search.predict(X_test)
    best_macro_average_F1 = f1_score(y_test, y_pred, average="macro")
    best_weighted_average_F1 = f1_score(y_test, y_pred, average="weighted")

    return best_n_estimators, best_max_leaf_nodes, best_cross_validation, best_test_set, best_macro_average_F1, best_weighted_average_F1


### Part 2: Results

In [14]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn).
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All results should be printed to 4 decimal places except for
# "k", "p", n_estimators" and "max_leaf_nodes" which should be printed as integers.
#cvKFold = StratifiedKFold(n_splits=10, shuffle=False)

from sklearn.model_selection import train_test_split

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

best_k, best_p, best_cross_validation, best_test_set = bestKNNClassifier(X_train, X_test, y_train, y_test)
print(best_cross_validation)
print(f"KNN best k: {best_k}")
print(f"KNN best p: {best_p}")
print(f"KNN cross-validation accuracy: {best_cross_validation:.4f}")
print(f"KNN test set accuracy: {best_test_set:.4f}")

print()

best_n_estimators, best_max_leaf_nodes, best_cross_validation, best_test_set, best_macro_average_F1, best_weighted_average_F1 = bestRFClassifier(X_train, X_test, y_train, y_test)

print(f"RF best n_estimators: {best_n_estimators}")
print(f"RF best max_leaf_nodes: {best_max_leaf_nodes}")
print(f"RF cross-validation accuracy: {best_cross_validation:.4f}")
print(f"RF test set accuracy: {best_test_set:.4f}")
print(f"RF test set macro average F1: {best_macro_average_F1:.4f}")
print(f"RF test set weighted average F1: {best_weighted_average_F1:.4f}")

0.9371428571428572
KNN best k: 5
KNN best p: 1
KNN cross-validation accuracy: 0.9371
KNN test set accuracy: 0.9257

RF best n_estimators: 10
RF best max_leaf_nodes: 12
RF cross-validation accuracy: 0.9400
RF test set accuracy: 0.9343
RF test set macro average F1: 0.9326
RF test set weighted average F1: 0.9341
